# Inferência — Private Test Set AH Challenge
Pré-processamento completo + Fusão B (janelas) → trial-0.txt

In [1]:
# CÉLULA 0: Instalar dependências
!pip install py-feat transformers librosa soundfile mediapipe

  Using cached numpy-1.23.5-cp310-cp310-win_amd64.whl.metadata (2.3 kB)
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
Using cached numpy-1.23.5-cp310-cp310-win_amd64.whl (14.6 MB)
   ---------------------------------------- 0.0/46.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/46.2 MB ? eta -:--:--
   --- ------------------------------------ 3.9/46.2 MB 16.8 MB/s eta 0:00:03
   ------- -------------------------------- 8.1/46.2 MB 18.6 MB/s eta 0:00:03
   --------- ------------------------------ 10.5/46.2 MB 15.9 MB/s eta 0:00:03
   -------------- ------------------------- 16.8/46.2 MB 19.6 MB/s eta 0:00:02
   ------------------------ --------------- 28.6/46.2 MB 27.0 MB/s eta 0:00:01
   -------------------------------- ------- 37.7/46.2 MB 32.0 MB/s eta 0:00:01
   ---------------------------------------  45.6/46.2 MB 30.9 MB/s eta 0:00:01
   --------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.23.5 which is incompatible.


In [2]:
# CÉLULA 1: Imports e configuração
import os
import re
import time
import subprocess
import numpy as np
import torch
import torch.nn as nn
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model, BertTokenizer, BertModel
from feat import Detector
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# === PATHS ===
# Dataset original (pra carregar stats de normalização e modelo treinado)
TRAIN_DIR = r'C:\Users\ddonz\OneDrive\Documentos\Aislan\data'

# Private test set
TEST_ROOT = r'C:\Users\ddonz\OneDrive\Documentos\Aislan\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\data\data'

# Outputs do pré-processamento
TEST_AU_DIR = os.path.join(TEST_ROOT, 'au_features')
TEST_AUDIO_WAV_DIR = os.path.join(TEST_ROOT, 'audio_wav')
TEST_AUDIO_DIR = os.path.join(TEST_ROOT, 'audio_features')

# Diretórios do test set
TEST_FACES_DIR = os.path.join(TEST_ROOT, 'cropped-aligned-faces', 'Videos')
TEST_VIDEOS_DIR = os.path.join(TEST_ROOT, 'Videos')
TEST_SPLIT_DIR = os.path.join(TEST_ROOT, 'split')
TEST_TRANSCRIPTION_DIR = os.path.join(TEST_ROOT, 'transcription')

os.makedirs(TEST_AU_DIR, exist_ok=True)
os.makedirs(TEST_AUDIO_WAV_DIR, exist_ok=True)
os.makedirs(TEST_AUDIO_DIR, exist_ok=True)

# Participantes excluídos
EXCLUDED_TEST_IDS = {'83246', '83249', '83277', '83301'}

SAMPLE_RATE_VISUAL = 3  # 1 a cada 3 frames

AU_COLS = [
    'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10',
    'AU11', 'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24',
    'AU25', 'AU26', 'AU28', 'AU43'
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Test root: {TEST_ROOT}")
print(f"Train dir (pra modelo): {TRAIN_DIR}")

Device: cuda
Test root: C:\Users\ddonz\OneDrive\Documentos\Aislan\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\data\data
Train dir (pra modelo): C:\Users\ddonz\OneDrive\Documentos\Aislan\data


In [3]:
# CÉLULA 2: Descobrir estrutura do test set

print("=== Conteúdo do test root ===")
for item in sorted(os.listdir(TEST_ROOT)):
    full = os.path.join(TEST_ROOT, item)
    if os.path.isdir(full):
        print(f"  [DIR] {item}")
    else:
        print(f"  [FILE] {item}")

# Descobrir vídeos
test_videos = []

# Opção 1: faces cropadas
if os.path.isdir(TEST_FACES_DIR):
    for pid in sorted(os.listdir(TEST_FACES_DIR)):
        if pid in EXCLUDED_TEST_IDS:
            continue
        visit_path = os.path.join(TEST_FACES_DIR, pid, 'Visite_1')
        if not os.path.isdir(visit_path):
            continue
        for video_name in sorted(os.listdir(visit_path)):
            v_path = os.path.join(visit_path, video_name)
            if os.path.isdir(v_path):
                test_videos.append({'pid': pid, 'video_name': video_name, 'faces_path': v_path})

print(f"\nVídeos encontrados: {len(test_videos)}")
print(f"Participantes: {len(set(v['pid'] for v in test_videos))}")
print(f"Excluídos: {EXCLUDED_TEST_IDS}")
if test_videos:
    print(f"Exemplo: {test_videos[0]}")

=== Conteúdo do test root ===
  [FILE] BAH_Dataset_EULA-2.pdf
  [DIR] Frames
  [DIR] Videos
  [DIR] au_features
  [DIR] audio_features
  [DIR] audio_wav
  [DIR] cropped-aligned-faces
  [FILE] documentation.pdf
  [FILE] extract_frames_from_videos.py
  [FILE] extract_frames_from_videos.sh
  [FILE] meta_data.yml
  [FILE] readme.md
  [DIR] split
  [DIR] split-frames
  [DIR] transcription
  [FILE] version.txt
  [FILE] video.csv
  [FILE] video_annotation_transcript.yaml

Vídeos encontrados: 151
Participantes: 28
Excluídos: {'83277', '83246', '83249', '83301'}
Exemplo: {'pid': '82778', 'video_name': '82778_Question_1_2024-12-07_21-01-09_Video.mp4', 'faces_path': 'C:\\Users\\ddonz\\OneDrive\\Documentos\\Aislan\\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\\data\\data\\cropped-aligned-faces\\Videos\\82778\\Visite_1\\82778_Question_1_2024-12-07_21-01-09_Video.mp4'}


In [4]:
# CÉLULA 3: Descobrir transcritos do test set
# Tentar múltiplas fontes possíveis

test_transcripts = {}  # video_name → texto

# Opção 1: arquivo de split com transcritos (como train.txt)
for split_file in ['test.txt', 'private_test.txt']:
    spath = os.path.join(TEST_SPLIT_DIR, split_file)
    if os.path.exists(spath):
        print(f"Encontrado split: {spath}")
        with open(spath, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(',', 2)
                if len(parts) >= 1:
                    vpath = parts[0]
                    vname = vpath.replace('\\', '/').split('/')[-1].replace('.mp4', '')
                    text = parts[2] if len(parts) > 2 else ''
                    # Pode ser que label seja -1 ou ausente no test
                    test_transcripts[vname] = text
        print(f"  Transcritos carregados: {len(test_transcripts)}")
        break

# Opção 2: pasta transcription/
if len(test_transcripts) == 0 and os.path.isdir(TEST_TRANSCRIPTION_DIR):
    print(f"Buscando em: {TEST_TRANSCRIPTION_DIR}")
    for root, dirs, files in os.walk(TEST_TRANSCRIPTION_DIR):
        for f in files:
            if f.endswith('.txt'):
                fpath = os.path.join(root, f)
                vname = f.replace('.txt', '').replace('.mp4', '')
                with open(fpath, 'r', encoding='utf-8') as fp:
                    text = fp.read().strip()
                test_transcripts[vname] = text
    print(f"  Transcritos de arquivos: {len(test_transcripts)}")

# Opção 3: YAML de anotações
if len(test_transcripts) == 0:
    import yaml
    for yaml_file in ['video_annotation_transcript.yaml', 'video_annotation_transcript.yml']:
        ypath = os.path.join(TEST_ROOT, yaml_file)
        if os.path.exists(ypath):
            print(f"Buscando em YAML: {ypath}")
            with open(ypath, 'r', encoding='utf-8') as f:
                annotations = yaml.safe_load(f)
            for vid, data in annotations.items():
                vname = vid.replace('\\', '/').split('/')[-1].replace('.mp4', '')
                transcript = data.get('transcript', {})
                if isinstance(transcript, dict):
                    text = transcript.get('text', '')
                else:
                    text = str(transcript)
                test_transcripts[vname] = text
            print(f"  Transcritos de YAML: {len(test_transcripts)}")
            break

# Verificar cobertura
matched = sum(1 for v in test_videos if v['video_name'] in test_transcripts 
              or v['video_name'].replace('.mp4', '') in test_transcripts)
print(f"\nVídeos com transcrito: {matched}/{len(test_videos)}")

if matched == 0:
    # Tentar match mais flexível
    print("\nTentando match flexível...")
    print(f"Primeiros video_names: {[v['video_name'] for v in test_videos[:3]]}")
    if test_transcripts:
        print(f"Primeiras keys transcritos: {list(test_transcripts.keys())[:3]}")

Encontrado split: C:\Users\ddonz\OneDrive\Documentos\Aislan\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\data\data\split\test.txt
  Transcritos carregados: 161

Vídeos com transcrito: 151/151


In [5]:
# CÉLULA 4: Extrair AUs do test set (Py-Feat)

def frame_sort_key(filename):
    match = re.search(r'frame-(\d+)', filename)
    return int(match.group(1)) if match else 0

# Checar quais já foram processados
au_to_process = []
au_already = 0
for v in test_videos:
    out_path = os.path.join(TEST_AU_DIR, v['pid'], f"{v['video_name']}.npy")
    if os.path.exists(out_path):
        au_already += 1
    else:
        au_to_process.append(v)

print(f"AUs: {au_already} já feitos, {len(au_to_process)} a processar")

if len(au_to_process) > 0:
    print("Inicializando Py-Feat...")
    pyfeat_detector = Detector(device="cuda")
    
    errors_au = []
    for idx, v in enumerate(tqdm(au_to_process, desc="Extraindo AUs")):
        pid_dir = os.path.join(TEST_AU_DIR, v['pid'])
        os.makedirs(pid_dir, exist_ok=True)
        out_path = os.path.join(pid_dir, f"{v['video_name']}.npy")
        
        frames = [f for f in os.listdir(v['faces_path']) if f.endswith(('.jpg', '.png'))]
        frames = sorted(frames, key=frame_sort_key)[::SAMPLE_RATE_VISUAL]
        frame_paths = [os.path.join(v['faces_path'], f) for f in frames]
        
        if len(frame_paths) == 0:
            errors_au.append(f"{v['pid']}/{v['video_name']}: sem frames")
            continue
        
        try:
            au_values = []
            for fp in frame_paths:
                try:
                    result = pyfeat_detector.detect_image(fp)
                    aus = result[AU_COLS].values[0].astype(np.float64)
                    au_values.append(aus)
                except:
                    au_values.append(np.zeros(20))
            
            au_array = np.array(au_values, dtype=np.float64)
            np.save(out_path, au_array)
        except Exception as e:
            errors_au.append(f"{v['pid']}/{v['video_name']}: {str(e)}")
    
    print(f"Concluído. Erros: {len(errors_au)}")
    if errors_au:
        for e in errors_au[:5]:
            print(f"  {e}")
else:
    print("Todas as AUs já extraídas!")

AUs: 151 já feitos, 0 a processar
Todas as AUs já extraídas!


In [6]:
# CÉLULA 5: Extrair áudio do test set (ffmpeg → wav → Wav2Vec 2.0)

# Passo 1: mp4 → wav
wav_to_process = []
wav_already = 0

for v in test_videos:
    # Encontrar .mp4
    mp4_path = os.path.join(TEST_VIDEOS_DIR, v['pid'], 'Visite_1', f"{v['video_name']}.mp4")
    if not os.path.exists(mp4_path):
        # Tentar sem .mp4 no video_name
        mp4_path = os.path.join(TEST_VIDEOS_DIR, v['pid'], 'Visite_1', v['video_name'])
        if not os.path.exists(mp4_path):
            continue
    
    wav_dir = os.path.join(TEST_AUDIO_WAV_DIR, v['pid'])
    wav_name = v['video_name'].replace('.mp4', '') + '.wav'
    wav_path = os.path.join(wav_dir, wav_name)
    
    if os.path.exists(wav_path):
        wav_already += 1
    else:
        wav_to_process.append({'v': v, 'mp4_path': mp4_path, 'wav_path': wav_path, 'wav_dir': wav_dir})

print(f"WAVs: {wav_already} já feitos, {len(wav_to_process)} a processar")

for item in tqdm(wav_to_process, desc="Extraindo WAV"):
    os.makedirs(item['wav_dir'], exist_ok=True)
    try:
        cmd = [
            'ffmpeg', '-i', item['mp4_path'],
            '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', '-y',
            item['wav_path']
        ]
        subprocess.run(cmd, capture_output=True, check=True)
    except Exception as e:
        print(f"  Erro: {item['v']['video_name']}: {e}")

# Passo 2: wav → Wav2Vec embeddings
embed_to_process = []
embed_already = 0

for v in test_videos:
    npy_dir = os.path.join(TEST_AUDIO_DIR, v['pid'])
    npy_name = v['video_name'].replace('.mp4', '') + '.npy'
    npy_path = os.path.join(npy_dir, npy_name)
    
    wav_name = v['video_name'].replace('.mp4', '') + '.wav'
    wav_path = os.path.join(TEST_AUDIO_WAV_DIR, v['pid'], wav_name)
    
    if os.path.exists(npy_path):
        embed_already += 1
    elif os.path.exists(wav_path):
        embed_to_process.append({'v': v, 'wav_path': wav_path, 'npy_path': npy_path, 'npy_dir': npy_dir})

print(f"\nEmbeddings: {embed_already} já feitos, {len(embed_to_process)} a processar")

if len(embed_to_process) > 0:
    print("Carregando Wav2Vec 2.0...")
    w2v_processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
    w2v_model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(device)
    w2v_model.eval()
    
    AUDIO_SR = 16000
    MAX_AUDIO_SECS = 60
    
    for item in tqdm(embed_to_process, desc="Wav2Vec embeddings"):
        os.makedirs(item['npy_dir'], exist_ok=True)
        try:
            audio, sr = librosa.load(item['wav_path'], sr=AUDIO_SR)
            if len(audio) > MAX_AUDIO_SECS * AUDIO_SR:
                audio = audio[:MAX_AUDIO_SECS * AUDIO_SR]
            
            inputs = w2v_processor(
                audio, sampling_rate=AUDIO_SR, return_tensors='pt', padding=True
            ).input_values.to(device)
            
            with torch.no_grad():
                outputs = w2v_model(inputs)
                embeddings = outputs.last_hidden_state.squeeze(0).cpu().numpy()
            
            np.save(item['npy_path'], embeddings.astype(np.float32))
        except Exception as e:
            print(f"  Erro: {item['v']['video_name']}: {e}")
    
    del w2v_model, w2v_processor
    torch.cuda.empty_cache()

print("Extração de áudio completa!")

WAVs: 0 já feitos, 151 a processar


Extraindo WAV:   0%|          | 0/151 [00:00<?, ?it/s]


Embeddings: 0 já feitos, 151 a processar
Carregando Wav2Vec 2.0...


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2Vec embeddings:   0%|          | 0/151 [00:00<?, ?it/s]

Extração de áudio completa!


In [7]:
# CÉLULA 6: Validação do pré-processamento

ready_videos = []
missing_au = 0
missing_audio = 0
missing_text = 0

for v in test_videos:
    vname = v['video_name']
    vname_clean = vname.replace('.mp4', '')
    
    # AU
    au_path = os.path.join(TEST_AU_DIR, v['pid'], f"{vname}.npy")
    if not os.path.exists(au_path):
        missing_au += 1
        continue
    
    # Audio
    audio_path = None
    for suffix in [f"{vname_clean}.npy", f"{vname}.npy"]:
        p = os.path.join(TEST_AUDIO_DIR, v['pid'], suffix)
        if os.path.exists(p):
            audio_path = p
            break
    if audio_path is None:
        missing_audio += 1
        continue
    
    # Texto
    text = ''
    for key in [vname, vname_clean, vname + '.mp4']:
        if key in test_transcripts:
            text = test_transcripts[key]
            break
    if not text:
        missing_text += 1
        # Continuar mesmo sem texto — usar string vazia
    
    # Video ID pra submission (formato: Videos/{pid}/Visite_1/{video_name}.mp4)
    video_id = f"Videos/{v['pid']}/Visite_1/{vname_clean}.mp4"
    
    ready_videos.append({
        'video_id': video_id,
        'pid': v['pid'],
        'video_name': vname,
        'au_path': au_path,
        'audio_path': audio_path,
        'text': text
    })

print(f"Prontos pra inferência: {len(ready_videos)}")
print(f"Missing AU: {missing_au} | Missing audio: {missing_audio} | Missing texto: {missing_text}")
if ready_videos:
    print(f"Exemplo video_id: {ready_videos[0]['video_id']}")

Prontos pra inferência: 151
Missing AU: 0 | Missing audio: 0 | Missing texto: 0
Exemplo video_id: Videos/82778/Visite_1/82778_Question_1_2024-12-07_21-01-09_Video.mp4


In [8]:
# CÉLULA 7: Definir modelo Fusão B (janelas) — mesma arquitetura do treino

WINDOW_SIZE = 16
WINDOW_STEP = 8
MAX_WINDOWS = 128
VISUAL_DIM = 80  # 4 stats × 20 AUs
AUDIO_MAX_LEN = 512
TEXT_MAX_LEN = 128


def compute_windows(au_seq, window_size=16, window_step=8):
    """Transforma série AU (N, 20) em janelas (W, 80)."""
    au_seq = au_seq.astype(np.float64)
    au_seq = np.nan_to_num(au_seq, nan=0.0, posinf=1.0, neginf=0.0)
    au_seq = np.clip(au_seq, 0.0, 5.0)
    
    n_aus = au_seq.shape[1]
    windows = []
    for start in range(0, len(au_seq) - window_size + 1, window_step):
        window = au_seq[start:start + window_size]
        feats = []
        for au_idx in range(n_aus):
            signal = window[:, au_idx]
            feats.extend([
                np.mean(signal),
                np.std(signal),
                np.polyfit(np.arange(window_size), signal, 1)[0],
                np.max(signal) - np.min(signal)
            ])
        windows.append(feats)
    
    if len(windows) == 0:
        return np.zeros((1, n_aus * 4))
    return np.array(windows)


def pad_or_truncate(seq, max_len):
    seq_len = len(seq)
    dim = seq.shape[1]
    if seq_len > max_len:
        indices = np.linspace(0, seq_len - 1, max_len, dtype=int)
        return seq[indices], np.ones(max_len)
    else:
        pad_len = max_len - seq_len
        padded = np.vstack([seq, np.zeros((pad_len, dim))])
        mask = np.concatenate([np.ones(seq_len), np.zeros(pad_len)])
        return padded, mask


class MultimodalAHDetector(nn.Module):
    def __init__(self, fusion_type='B', visual_dim=80, hidden_dim=128,
                 dropout=0.3, freeze_bert=True):
        super().__init__()
        self.fusion_type = fusion_type
        proj_dim = hidden_dim
        
        # Visual encoder
        self.visual_lstm = nn.LSTM(
            input_size=visual_dim, hidden_size=hidden_dim,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=dropout
        )
        self.visual_attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.Tanh(), nn.Linear(64, 1)
        )
        self.visual_proj = nn.Linear(hidden_dim * 2, proj_dim)
        
        # Audio encoder
        self.audio_input_proj = nn.Linear(768, hidden_dim)
        self.audio_lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=dropout
        )
        self.audio_attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.Tanh(), nn.Linear(64, 1)
        )
        self.audio_proj = nn.Linear(hidden_dim * 2, proj_dim)
        
        # Text encoder
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
            for param in self.bert.encoder.layer[-2:].parameters():
                param.requires_grad = True
        self.text_proj = nn.Linear(768, proj_dim)
        
        # Classifier — Fusão B
        cls_input = proj_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(cls_input, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def _attend(self, lstm_out, mask, attn_layer):
        weights = attn_layer(lstm_out).squeeze(-1)
        weights = weights.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(weights, dim=1)
        return torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
    
    def forward(self, visual, v_mask, audio, a_mask, input_ids, attention_mask):
        # Visual
        v_out, _ = self.visual_lstm(visual)
        h_v = self._attend(v_out, v_mask, self.visual_attn)
        h_v = self.visual_proj(h_v)
        
        # Audio
        a_proj = self.audio_input_proj(audio)
        a_out, _ = self.audio_lstm(a_proj)
        h_a = self._attend(a_out, a_mask, self.audio_attn)
        h_a = self.audio_proj(h_a)
        
        # Text
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h_t = self.text_proj(bert_out.last_hidden_state[:, 0])
        
        # Fusão B: divergências
        d_va = torch.abs(h_v - h_a)
        d_vt = torch.abs(h_v - h_t)
        d_at = torch.abs(h_a - h_t)
        fused = torch.cat([d_va, d_vt, d_at], dim=1)
        
        return self.classifier(fused)


print("Modelo definido.")

Modelo definido.


In [9]:
# CÉLULA 8: Carregar modelo treinado e stats de normalização
# IMPORTANTE: precisa ter rodado o multimodal_fusion_windowed.ipynb antes
# e ter o arquivo best_win_fusion_B.pt salvo

MODEL_PATH = os.path.join(TRAIN_DIR, 'best_win_fusion_B.pt')

print(f"Carregando modelo de: {MODEL_PATH}")
assert os.path.exists(MODEL_PATH), f"Modelo não encontrado em {MODEL_PATH}!"

model = MultimodalAHDetector(fusion_type='B', visual_dim=VISUAL_DIM).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Modelo carregado!")

# Calcular stats de normalização visual a partir dos dados de TREINO
# (mesma normalização usada no treino)
print("\nCalculando stats de normalização do treino...")

TRAIN_SPLIT = os.path.join(TRAIN_DIR, 'split', 'train.txt')
TRAIN_AU_DIR = os.path.join(TRAIN_DIR, 'au_features')

EXCLUDED_TRAIN_IDS = set(str(x) for x in [
    82723, 82687, 82569, 82570, 82576, 82577, 82581, 82587, 82589,
    82624, 82627, 82628, 82642, 82652, 82664, 82665, 82674, 82677,
    82681, 82690, 82705, 82708, 82709, 82738, 82758, 82768,
    82777, 82783, 82784, 82794, 82807, 82812, 82813, 82814,
    82815, 82817, 82819, 82820, 82832, 82845, 82861, 82866,
    82875, 82879, 82895, 82899, 82910, 82912, 82919, 82555,
    82786, 82827, 82927, 82928, 82956, 82968, 83008, 83011,
    83045, 83080, 83086
])

all_train_windows = []
with open(TRAIN_SPLIT, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(',', 2)
        if len(parts) < 2:
            continue
        path_parts = parts[0].replace('\\', '/').split('/')
        pid = path_parts[1]
        vname = path_parts[-1].replace('.mp4', '')
        
        if pid in EXCLUDED_TRAIN_IDS:
            continue
        
        au_path = None
        for suffix in [f'{vname}.mp4.npy', f'{vname}.npy']:
            p = os.path.join(TRAIN_AU_DIR, pid, suffix)
            if os.path.exists(p):
                au_path = p
                break
        if au_path is None:
            continue
        
        au_seq = np.load(au_path).astype(np.float64)
        au_seq = np.nan_to_num(au_seq, nan=0.0)
        au_seq = np.clip(au_seq, 0.0, 5.0)
        
        if len(au_seq) >= WINDOW_SIZE:
            windows = compute_windows(au_seq, WINDOW_SIZE, WINDOW_STEP)
            all_train_windows.append(windows)

all_w = np.vstack(all_train_windows)
v_mean = np.mean(all_w, axis=0)
v_std = np.std(all_w, axis=0)
v_std[v_std < 1e-8] = 1.0

print(f"Stats calculadas a partir de {len(all_train_windows)} vídeos de treino")
print(f"v_mean shape: {v_mean.shape}, v_std shape: {v_std.shape}")

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("Tokenizer pronto.")

Carregando modelo de: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\best_win_fusion_B.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado!

Calculando stats de normalização do treino...
Stats calculadas a partir de 598 vídeos de treino
v_mean shape: (80,), v_std shape: (80,)
Tokenizer pronto.


In [10]:
# CÉLULA 9: Inferência — gerar predições

predictions = []  # lista de (video_id, prob_0, prob_1, label)

model.eval()

for v in tqdm(ready_videos, desc="Inferência"):
    try:
        # === Visual: AU → janelas → normalizar → pad/truncate ===
        au_seq = np.load(v['au_path']).astype(np.float64)
        au_seq = np.nan_to_num(au_seq, nan=0.0)
        au_seq = np.clip(au_seq, 0.0, 5.0)
        
        if len(au_seq) < WINDOW_SIZE:
            # Pad a sequência pra ter pelo menos 1 janela
            pad_len = WINDOW_SIZE - len(au_seq)
            au_seq = np.vstack([au_seq, np.zeros((pad_len, 20))])
        
        windows = compute_windows(au_seq, WINDOW_SIZE, WINDOW_STEP)
        windows = (windows - v_mean) / v_std
        v_padded, v_mask = pad_or_truncate(windows, MAX_WINDOWS)
        
        # === Áudio ===
        audio_seq = np.load(v['audio_path']).astype(np.float32)
        a_padded, a_mask = pad_or_truncate(audio_seq, AUDIO_MAX_LEN)
        
        # === Texto ===
        encoding = tokenizer(
            v['text'], max_length=TEXT_MAX_LEN,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        
        # === Preparar tensores ===
        v_tensor = torch.FloatTensor(v_padded).unsqueeze(0).to(device)
        vm_tensor = torch.FloatTensor(v_mask).unsqueeze(0).to(device)
        a_tensor = torch.FloatTensor(a_padded).unsqueeze(0).to(device)
        am_tensor = torch.FloatTensor(a_mask).unsqueeze(0).to(device)
        ids_tensor = encoding['input_ids'].to(device)
        att_tensor = encoding['attention_mask'].to(device)
        
        # === Forward ===
        with torch.no_grad():
            logit = model(v_tensor, vm_tensor, a_tensor, am_tensor, ids_tensor, att_tensor)
            prob_1 = torch.sigmoid(logit).item()
            prob_0 = 1.0 - prob_1
            label = 1 if prob_1 > 0.5 else 0
        
        predictions.append({
            'video_id': v['video_id'],
            'prob_0': prob_0,
            'prob_1': prob_1,
            'label': label
        })
    
    except Exception as e:
        print(f"  Erro em {v['video_id']}: {e}")
        # Predição default
        predictions.append({
            'video_id': v['video_id'],
            'prob_0': 0.5,
            'prob_1': 0.5,
            'label': 0
        })

print(f"\nPredições geradas: {len(predictions)}")
n_ah = sum(1 for p in predictions if p['label'] == 1)
print(f"Predições A/H=1: {n_ah} ({n_ah/len(predictions)*100:.1f}%)")
print(f"Predições A/H=0: {len(predictions)-n_ah} ({(len(predictions)-n_ah)/len(predictions)*100:.1f}%)")

Inferência:   0%|          | 0/151 [00:00<?, ?it/s]


Predições geradas: 151
Predições A/H=1: 33 (21.9%)
Predições A/H=0: 118 (78.1%)


In [11]:
# CÉLULA 10: Gerar trial-0.txt

OUTPUT_PATH = os.path.join(TEST_ROOT, 'trial-0.txt')

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for p in predictions:
        f.write(f"{p['video_id']},{p['prob_0']:.6f},{p['prob_1']:.6f},{p['label']}\n")

print(f"Arquivo salvo: {OUTPUT_PATH}")
print(f"Total linhas: {len(predictions)}")

# Mostrar primeiras e últimas linhas
print("\nPrimeiras 5 linhas:")
with open(OUTPUT_PATH, 'r') as f:
    lines = f.readlines()
    for line in lines[:5]:
        print(f"  {line.strip()}")
    print(f"  ...")
    for line in lines[-3:]:
        print(f"  {line.strip()}")

Arquivo salvo: C:\Users\ddonz\OneDrive\Documentos\Aislan\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\data\data\trial-0.txt
Total linhas: 151

Primeiras 5 linhas:
  Videos/82778/Visite_1/82778_Question_1_2024-12-07_21-01-09_Video.mp4,0.969324,0.030676,0
  Videos/82778/Visite_1/82778_Question_2_2024-12-07_21-05-46_Video.mp4,0.637401,0.362599,0
  Videos/82778/Visite_1/82778_Question_3_2024-12-07_21-04-30_Video.mp4,0.970587,0.029413,0
  Videos/82778/Visite_1/82778_Question_4_2024-12-07_21-07-19_Video.mp4,0.199936,0.800064,1
  Videos/82778/Visite_1/82778_Question_5_2024-12-07_21-01-56_Video.mp4,0.690358,0.309642,0
  ...
  Videos/83278/Visite_1/83278_Question_1_2025-08-30_13-21-00_Video.mp4,0.986207,0.013793,0
  Videos/83278/Visite_1/83278_Question_2_2025-08-30_13-22-05_Video.mp4,0.883061,0.116939,0
  Videos/83278/Visite_1/83278_Question_7_2025-08-30_13-27-58_Video.mp4,0.134427,0.865573,1


In [13]:
# CÉLULA 11: Resumo e próximos passos

print("=" * 60)
print("RESUMO DA SUBMISSÃO")
print("=" * 60)
print(f"Modelo: Fusão B (divergência) com visual por janelas")
print(f"Visual: 20 AUs → sliding window (W={WINDOW_SIZE}, step={WINDOW_STEP}) → BiLSTM")
print(f"Áudio: Wav2Vec 2.0 → BiLSTM")
print(f"Texto: BERT (fine-tuned)")
print(f"Fusão: |h_v - h_a|, |h_v - h_t|, |h_a - h_t| → MLP")
print(f"")
print(f"Vídeos processados: {len(predictions)}")
print(f"Predições A/H: {n_ah}/{len(predictions)}")
print(f"Arquivo: {OUTPUT_PATH}")
print(f"")


RESUMO DA SUBMISSÃO
Modelo: Fusão B (divergência) com visual por janelas
Visual: 20 AUs → sliding window (W=16, step=8) → BiLSTM
Áudio: Wav2Vec 2.0 → BiLSTM
Texto: BERT (fine-tuned)
Fusão: |h_v - h_a|, |h_v - h_t|, |h_a - h_t| → MLP

Vídeos processados: 151
Predições A/H: 33/151
Arquivo: C:\Users\ddonz\OneDrive\Documentos\Aislan\abaw-10th-ah-cvpr-2026-bah-private-test-set-30-participants-unlabeled-public-access\data\data\trial-0.txt

